In [92]:
import torch
import torchvision
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn as nn
from torchvision.transforms import v2

In [93]:
trainingset = torchvision.datasets.FashionMNIST(
root="./data",
train=True,
download=True,
transform = v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True)])

)

In [94]:
testset = torchvision.datasets.FashionMNIST(
root="./data",
train=False, 
download=True,
transform = v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True)])
)

In [95]:
device = "cpu"

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,32,3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(1600,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self, x):
        return self.conv(x)

model = Net().to(device)
print(model)
    


Net(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1600, out_features=128, bias=True)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [96]:
batch_size = 64
Loaded_training_set = DataLoader(trainingset, batch_size=batch_size, shuffle=True)
Loaded_test_set = DataLoader(testset, batch_size= batch_size, shuffle = True)


In [97]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-2)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0 :
            loss, current = loss.item(), batch * len(X)
            print(f"loss is {loss:>7.4f} [{current}/{size:>5d}]")





In [98]:
def test(dataloader, model, loss_fn):
    num_batch = len(dataloader)
    size = len(dataloader.dataset)
    model.eval()
    test_loss , correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batch
    correct /= size
    print(f"test loss is {(test_loss):>0.1f} \n accuracy is {correct:>8f} ")


In [99]:
epoch = 20
for i in range(epoch):
    print(f"Epoch:{i+1}, \n -------------------")
    train(Loaded_training_set,model,loss_fn, optimizer) 
    test(Loaded_test_set,model,loss_fn)

Epoch:1, 
 -------------------
loss is  2.3069 [0/60000]
loss is  2.2705 [6400/60000]
loss is  2.1739 [12800/60000]
loss is  1.7759 [19200/60000]
loss is  1.0839 [25600/60000]
loss is  0.8978 [32000/60000]
loss is  0.8212 [38400/60000]
loss is  0.8135 [44800/60000]
loss is  0.6669 [51200/60000]
loss is  0.8925 [57600/60000]
test loss is 0.8 
 accuracy is 0.686400 
Epoch:2, 
 -------------------
loss is  0.9578 [0/60000]
loss is  0.9628 [6400/60000]
loss is  0.7147 [12800/60000]
loss is  0.7039 [19200/60000]
loss is  0.4239 [25600/60000]
loss is  0.7285 [32000/60000]
loss is  0.6880 [38400/60000]
loss is  0.7028 [44800/60000]
loss is  0.5630 [51200/60000]
loss is  0.8233 [57600/60000]
test loss is 0.7 
 accuracy is 0.737100 
Epoch:3, 
 -------------------
loss is  0.6822 [0/60000]
loss is  0.7036 [6400/60000]
loss is  0.7377 [12800/60000]
loss is  0.5572 [19200/60000]
loss is  0.6191 [25600/60000]
loss is  0.6290 [32000/60000]
loss is  0.6129 [38400/60000]
loss is  0.5360 [44800/60000]


In [ ]:
torch.save(model.state_dict(),"models/model_fashion.pth")

In [104]:
classes = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"] 

model.eval()
counter = 0
k = 35
for n in range(k):
    x,y = testset[n][0], testset[n][1]
    with torch.no_grad():
        x = x.unsqueeze(0).to(device)
        pred = model(x)
        predicted, actual = classes[pred[0].argmax(0)], classes[y]
        print(f'Predicted: "{predicted}", Actual: "{actual}"')
        if predicted == actual:
            counter += 1
print(f"Accuracy: {counter/k*100}%")

Predicted: "Ankle boot", Actual: "Ankle boot"
Predicted: "Pullover", Actual: "Pullover"
Predicted: "Trouser", Actual: "Trouser"
Predicted: "Trouser", Actual: "Trouser"
Predicted: "Shirt", Actual: "Shirt"
Predicted: "Trouser", Actual: "Trouser"
Predicted: "Coat", Actual: "Coat"
Predicted: "Shirt", Actual: "Shirt"
Predicted: "Sandal", Actual: "Sandal"
Predicted: "Sneaker", Actual: "Sneaker"
Predicted: "Coat", Actual: "Coat"
Predicted: "Sandal", Actual: "Sandal"
Predicted: "Sandal", Actual: "Sneaker"
Predicted: "Dress", Actual: "Dress"
Predicted: "Coat", Actual: "Coat"
Predicted: "Trouser", Actual: "Trouser"
Predicted: "Pullover", Actual: "Pullover"
Predicted: "Coat", Actual: "Coat"
Predicted: "Bag", Actual: "Bag"
Predicted: "T-shirt/top", Actual: "T-shirt/top"
Predicted: "Pullover", Actual: "Pullover"
Predicted: "Sandal", Actual: "Sandal"
Predicted: "Sneaker", Actual: "Sneaker"
Predicted: "Sandal", Actual: "Ankle boot"
Predicted: "Trouser", Actual: "Trouser"
Predicted: "Pullover", Actual